## Introduccion al Parser

El parser de pseudocodigo es el componente central del modulo de parsing. Su funcion principal es:

1. **Leer codigo pseudocodigo** en sintaxis estilo PSeInt
2. **Generar un Parse Tree** utilizando la gramatica definida en Lark
3. **Construir el AST** mediante el `ASTBuilder`

### Arquitectura del Parser

```
Codigo Pseudocodigo
        |
        v
+-------------------+
|  PseudocodeParser |
+-------------------+
        |
        v
+-------------------+
|   Lark Grammar    |
|  (pseudocode.lark)|
+-------------------+
        |
        v
+-------------------+
|   Parse Tree      |
+-------------------+
        |
        v
+-------------------+
|   ASTBuilder      |
+-------------------+
        |
        v
+-------------------+
|   AST (Nodos)     |
+-------------------+
```

## Componentes del Sistema de Parsing

### 1. PseudocodeParser

Ubicacion: `app/core/parser/pseudocode_parser.py`

Es la clase principal que expone el metodo `parse()`. Internamente:

- Carga la gramatica desde `grammar/pseudocode.lark`
- Utiliza el parser LALR de Lark (mas eficiente que Earley)
- Propaga informacion de posicion (linea/columna) para mensajes de error

### 2. ASTBuilder

Ubicacion: `app/core/parser/ast_builder.py`

Implementa el patron Transformer de Lark:

- Cada metodo corresponde a una regla de la gramatica
- Convierte tokens y subarboles en nodos AST tipados
- Maneja la construccion de estructuras anidadas

### 3. AST Nodes

Ubicacion: `app/core/parser/ast_nodes.py`

Define los tipos de nodos del AST:

| Nodo | Descripcion |
|------|-------------|
| `ProgramNode` | Raiz del programa |
| `AlgorithmNode` | Definicion del algoritmo |
| `BlockNode` | Bloque begin...end |
| `ForLoopNode` | Ciclo for |
| `WhileLoopNode` | Ciclo while |
| `RepeatLoopNode` | Ciclo repeat...until |
| `IfStatementNode` | Condicional if...then...else |
| `AssignmentNode` | Asignacion variable <- expresion |
| `CallStatementNode` | Llamada a funcion/procedimiento |
| `ReturnStatementNode` | Retorno de valor |

## Sintaxis Soportada

El parser soporta la siguiente sintaxis basada en PSeInt:

### Estructura Basica del Algoritmo

```
algorithm nombre_algoritmo(parametros)
begin
    // sentencias
end
```

### Palabras Clave Soportadas

| Categoria | Ingles | Espanol |
|-----------|--------|----------|
| Algoritmo | `algorithm` | `algoritmo`, `proceso`, `funcion` |
| Inicio | `begin` | `inicio` |
| Fin | `end` | `fin`, `finproceso`, `finalgoritmo` |
| Para | `for` | `para` |
| Hasta | `to` | `hasta` |
| Hacer | `do` | `hacer` |
| Mientras | `while` | `mientras` |
| Repetir | `repeat` | `repetir` |
| Hasta que | `until` | `hasta que` |
| Si | `if` | `si` |
| Entonces | `then` | `entonces` |
| Sino | `else` | `sino` |
| Llamar | `call` | `llamar` |
| Retornar | `return` | `retornar`, `devolver` |

### Operadores

| Tipo | Operadores |
|------|------------|
| Asignacion | `<-`, `:=` |
| Aritmeticos | `+`, `-`, `*`, `/`, `^`, `%` |
| Comparacion | `<`, `>`, `<=`, `>=`, `=`, `!=` |
| Logicos | `and`, `or`, `not` |

### Funciones Especiales

- `ceil()` o `techo()` - Techo matematico
- `floor()` o `piso()` - Piso matematico
- `length()` o `longitud()` - Longitud de arreglo

## Ejemplos de Uso

### Ejemplo 1: Algoritmo Simple con Ciclo For

El siguiente codigo representa un algoritmo de suma simple:

```
algorithm suma(n)
begin
    total <- 0
    for i <- 1 to n do
        total <- total + i
    end
    return total
end
```

Este codigo genera un AST con:
- Un `AlgorithmNode` con nombre "suma" y parametro "n"
- Un `BlockNode` conteniendo las sentencias
- Un `AssignmentNode` para `total <- 0`
- Un `ForLoopNode` con variable "i", inicio 1, fin "n"
- Un `ReturnStatementNode`

### Ejemplo 2: Algoritmo con Condicion

```
algorithm maximo(a, b)
begin
    if (a > b) then
        max <- a
    else
        max <- b
    end
    return max
end
```

### Ejemplo 3: Algoritmo Recursivo (MergeSort)

```
algorithm mergeSort(A[], p, r)
begin
    if (p < r) then
        q <- floor((p + r) / 2)
        call mergeSort(A, p, q)
        call mergeSort(A, q + 1, r)
        call merge(A, p, q, r)
    end
end
```

Este es un ejemplo de un algoritmo divide y venceras que el parser reconoce correctamente,
incluyendo:
- Parametros de tipo arreglo (`A[]`)
- Llamadas recursivas (`call mergeSort`)
- Uso de funciones matematicas (`floor`)

## Manejo de Errores

El parser proporciona mensajes de error descriptivos:

### Tipos de Excepciones

| Excepcion | Descripcion |
|-----------|-------------|
| `SyntaxErrorException` | Token inesperado en el codigo |
| `TokenizationException` | Caracter no reconocido |
| `ParserException` | Error general del parser |
| `ASTBuildException` | Error al construir el AST |

### Informacion de Error

Los errores incluyen:
- Numero de linea y columna
- Token encontrado vs esperado
- Contexto del error

### Errores Comunes

1. **Falta de `end`**: Olvidar cerrar un bloque
2. **Operador de asignacion incorrecto**: Usar `=` en lugar de `<-`
3. **Parentesis desbalanceados**: En condiciones o expresiones
4. **Identificador invalido**: Nombres que empiezan con numeros

## Demostracion: Uso del Parser

A continuacion se muestra como utilizar el parser programaticamente:

In [ ]:
import sys
sys.path.insert(0, '../..')

from app.core.parser.pseudocode_parser import PseudocodeParser

# Crear instancia del parser
parser = PseudocodeParser()

# Codigo de ejemplo
codigo = '''
algorithm bubbleSort(A[], n)
begin
    for i <- 1 to n - 1 do
        for j <- 1 to n - i do
            if (A[j] > A[j + 1]) then
                temp <- A[j]
                A[j] <- A[j + 1]
                A[j + 1] <- temp
            end
        end
    end
end
'''

# Parsear el codigo
ast = parser.parse(codigo)

# Mostrar informacion del AST
print(f"Nombre del algoritmo: {ast.algorithm.name}")
print(f"Numero de parametros: {len(ast.algorithm.parameters)}")
print(f"Parametros: {[p.name for p in ast.algorithm.parameters]}")

---

## Conclusiones

El parser de pseudocodigo proporciona:

1. **Flexibilidad**: Soporta sintaxis en ingles y espanol
2. **Robustez**: Manejo completo de errores con mensajes descriptivos
3. **Estructura clara**: AST bien tipado para analisis posteriores
4. **Eficiencia**: Parser LALR para rendimiento optimo

El AST generado es utilizado por los modulos de:
- Analisis de complejidad (Big O, Omega, Theta)
- Deteccion de patrones algoritmicos
- Generacion de visualizaciones

---

**Siguiente notebook recomendado**: `ast_visualization.ipynb` para visualizar la estructura del AST generado.